In [ ]:
import pandas as pd
from pathlib import Path
import requests
from PIL import Image
from io import BytesIO
from transformers import pipeline

# =====================================================================
# 1. RUTAS Y DETECCIÓN DE DATOS
# =====================================================================
DIRECTORIO_ACTUAL = Path.cwd()
if DIRECTORIO_ACTUAL.name in ["src", "notebooks"]:
    DIRECTORIO_RAIZ = DIRECTORIO_ACTUAL.parent
else:
    DIRECTORIO_RAIZ = DIRECTORIO_ACTUAL

DIRECTORIO_DATOS = DIRECTORIO_RAIZ / "data"

# =====================================================================
# 2. CARGA DEL MODELO DE VISIÓN ARTIFICIAL (CLIP)
# =====================================================================
print("Cargando modelo de Visión Artificial...")
clasificador = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

ETIQUETAS_BUSQUEDA = [
    "a swimming pool",          
    "a modern living room",     
    "a terrace or balcony",     
    "a bright and sunny room"   
]

def analizar_imagen_airbnb(url):
    try:
        respuesta = requests.get(url, timeout=5)
        imagen = Image.open(BytesIO(respuesta.content)).convert("RGB")
        
        predicciones = clasificador(imagen, candidate_labels=ETIQUETAS_BUSQUEDA)
        
        resultados = {}
        for pred in predicciones:
            etiqueta = pred['label'].replace(" ", "_")
            resultados[f"img_has_{etiqueta}"] = 1 if pred['score'] > 0.25 else 0
            
        return resultados
    except Exception:
        return {f"img_has_{etiq.replace(' ', '_')}": None for etiq in ETIQUETAS_BUSQUEDA}

# =====================================================================
# 3. PROCESAMIENTO DE LAS FOTOS POR AÑO
# =====================================================================
for año in [2025, 2026]:
    ruta_archivo = DIRECTORIO_DATOS / str(año) / "listings.csv.gz"
    ruta_salida = DIRECTORIO_DATOS / f"features_imagenes_{año}.csv"
    
    if not ruta_archivo.exists():
        print(f" No se encontró el archivo {ruta_archivo}")
        continue
        
    print(f"\n Cargando datos detallados de {año}...")
    # Cargamos el CSV comprimido que contiene la columna 'picture_url'
    df_listings = pd.read_csv(ruta_archivo, compression='gzip', low_memory=False)
    
    # Nos aseguramos de que existe la columna de fotos
    if 'picture_url' not in df_listings.columns:
        print(f" El archivo de {año} no tiene la columna 'picture_url'.")
        continue

    print(f" Iniciando análisis de imágenes para {año}...")
    resultados_imagenes = []
    
    # ATENCIÓN: para procesar los miles de pisos (¡tardará varias horas!)
    # Por ello, se usa .head(50) para probar que funciona.
    # Cuando se compruebe que el archivo se genera correctamente, quitar el .head(50) 
    
    for index, fila in df_listings.head(50).iterrows():
        id_anuncio = fila['id']
        url = fila['picture_url']
        
        # Mostramos progreso cada 10 imágenes
        if (index + 1) % 10 == 0:
            print(f"   Procesadas {index + 1} imágenes...")
            
        features_foto = analizar_imagen_airbnb(url)
        features_foto['listing_id'] = id_anuncio
        
        resultados_imagenes.append(features_foto)

    # Convertimos a DataFrame y guardamos el CSV final
    df_imagenes = pd.DataFrame(resultados_imagenes)
    df_imagenes.to_csv(ruta_salida, index=False)
    print(f" ¡Análisis de {año} guardado con éxito en: {ruta_salida}")

Cargando modelo de Visión Artificial...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


 Cargando datos detallados de 2025...
 Iniciando análisis de imágenes para 2025...
   Procesadas 10 imágenes...
   Procesadas 20 imágenes...
   Procesadas 30 imágenes...
   Procesadas 40 imágenes...
   Procesadas 50 imágenes...
   Procesadas 60 imágenes...
   Procesadas 70 imágenes...
   Procesadas 80 imágenes...
   Procesadas 90 imágenes...
   Procesadas 100 imágenes...
   Procesadas 110 imágenes...
   Procesadas 120 imágenes...
   Procesadas 130 imágenes...
   Procesadas 140 imágenes...
   Procesadas 150 imágenes...
   Procesadas 160 imágenes...
   Procesadas 170 imágenes...
   Procesadas 180 imágenes...
   Procesadas 190 imágenes...
   Procesadas 200 imágenes...
   Procesadas 210 imágenes...
   Procesadas 220 imágenes...
   Procesadas 230 imágenes...
   Procesadas 240 imágenes...
   Procesadas 250 imágenes...
   Procesadas 260 imágenes...
   Procesadas 270 imágenes...
   Procesadas 280 imágenes...
   Procesadas 290 imágenes...
   Procesadas 300 imágenes...
   Procesadas 310 imágene

c:\Users\Marilo-TURBO\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


   Procesadas 5090 imágenes...
   Procesadas 5100 imágenes...
   Procesadas 5110 imágenes...
   Procesadas 5120 imágenes...
   Procesadas 5130 imágenes...
   Procesadas 5140 imágenes...
   Procesadas 5150 imágenes...
   Procesadas 5160 imágenes...
   Procesadas 5170 imágenes...
   Procesadas 5180 imágenes...
   Procesadas 5190 imágenes...
   Procesadas 5200 imágenes...
   Procesadas 5210 imágenes...
   Procesadas 5220 imágenes...
   Procesadas 5230 imágenes...
   Procesadas 5240 imágenes...
   Procesadas 5250 imágenes...
   Procesadas 5260 imágenes...
   Procesadas 5270 imágenes...
   Procesadas 5280 imágenes...
   Procesadas 5290 imágenes...
   Procesadas 5300 imágenes...
   Procesadas 5310 imágenes...
   Procesadas 5320 imágenes...
   Procesadas 5330 imágenes...
   Procesadas 5340 imágenes...
   Procesadas 5350 imágenes...
   Procesadas 5360 imágenes...
   Procesadas 5370 imágenes...
   Procesadas 5380 imágenes...
   Procesadas 5390 imágenes...
   Procesadas 5400 imágenes...
   Proce

c:\Users\Marilo-TURBO\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


   Procesadas 4540 imágenes...
   Procesadas 4550 imágenes...
   Procesadas 4560 imágenes...
   Procesadas 4570 imágenes...
   Procesadas 4580 imágenes...
   Procesadas 4590 imágenes...
   Procesadas 4600 imágenes...
   Procesadas 4610 imágenes...
   Procesadas 4620 imágenes...
   Procesadas 4630 imágenes...
   Procesadas 4640 imágenes...
   Procesadas 4650 imágenes...
   Procesadas 4660 imágenes...
   Procesadas 4670 imágenes...
   Procesadas 4680 imágenes...
   Procesadas 4690 imágenes...
   Procesadas 4700 imágenes...
   Procesadas 4710 imágenes...
   Procesadas 4720 imágenes...
   Procesadas 4730 imágenes...
   Procesadas 4740 imágenes...
   Procesadas 4750 imágenes...
   Procesadas 4760 imágenes...
   Procesadas 4770 imágenes...
   Procesadas 4780 imágenes...
   Procesadas 4790 imágenes...
   Procesadas 4800 imágenes...
   Procesadas 4810 imágenes...
   Procesadas 4820 imágenes...
   Procesadas 4830 imágenes...
   Procesadas 4840 imágenes...
   Procesadas 4850 imágenes...
   Proce

c:\Users\Marilo-TURBO\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:3578: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


   Procesadas 6010 imágenes...
   Procesadas 6020 imágenes...
   Procesadas 6030 imágenes...
   Procesadas 6040 imágenes...
   Procesadas 6050 imágenes...
   Procesadas 6060 imágenes...
   Procesadas 6070 imágenes...
   Procesadas 6080 imágenes...
   Procesadas 6090 imágenes...
   Procesadas 6100 imágenes...
   Procesadas 6110 imágenes...
   Procesadas 6120 imágenes...
   Procesadas 6130 imágenes...
   Procesadas 6140 imágenes...
   Procesadas 6150 imágenes...
   Procesadas 6160 imágenes...
   Procesadas 6170 imágenes...
   Procesadas 6180 imágenes...
   Procesadas 6190 imágenes...
   Procesadas 6200 imágenes...
   Procesadas 6210 imágenes...
   Procesadas 6220 imágenes...
   Procesadas 6230 imágenes...
   Procesadas 6240 imágenes...
   Procesadas 6250 imágenes...
   Procesadas 6260 imágenes...
   Procesadas 6270 imágenes...
   Procesadas 6280 imágenes...
   Procesadas 6290 imágenes...
   Procesadas 6300 imágenes...
   Procesadas 6310 imágenes...
   Procesadas 6320 imágenes...
   Proce